In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Walker2d-v4 PPO Training with PyTorch
# Full RL implementation with comprehensive tracking and visualization

# ## 1. Install Dependencies
get_ipython().system('pip install "gymnasium[mujoco]" -q')
get_ipython().system('pip install imageio imageio-ffmpeg -q')
get_ipython().system('pip install matplotlib seaborn -q')

In [ ]:
# ## 2. Imports and Setup
import os
os.environ['MUJOCO_GL'] = 'egl'

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import seaborn as sns
from collections import deque
import json
import os
from datetime import datetime
import imageio
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Create directories
BASE_PATH = Path("/kaggle/working")
CHECKPOINT_DIR = BASE_PATH / "checkpoints"
VIDEO_DIR = BASE_PATH / "videos"
PLOT_DIR = BASE_PATH / "plots"
BEST_MODEL_DIR = BASE_PATH / "best_model"

for dir_path in [CHECKPOINT_DIR, VIDEO_DIR, PLOT_DIR, BEST_MODEL_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

In [ ]:
# ## 3. Hyperparameters
class Config:
    # Environment
    env_name = "Walker2d-v4"
    max_episodes = 3000
    max_timesteps = 1000
    
    # PPO Hyperparameters (cross-verified with CleanRL/SpinningUp)
    lr = 3e-4
    gamma = 0.99
    gae_lambda = 0.95
    clip_epsilon = 0.2
    clip_epsilon_decay = 0.999
    min_clip_epsilon = 0.1
    
    value_coef = 0.5
    entropy_coef = 0.01
    entropy_decay = 0.995
    min_entropy_coef = 0.001
    
    ppo_epochs = 10
    mini_batch_size = 64
    rollout_buffer_size = 2048
    
    # Network
    hidden_dim = 256
    
    # Logging & Checkpointing
    save_checkpoint_every = 100
    save_video_every = 250
    log_every = 10
    moving_avg_window = 100
    
    # Early stopping
    target_reward = 3000  # Walker2d-v4 solve threshold
    patience = 200  # episodes without improvement

config = Config()

print("\n=== Configuration ===")
for key, value in vars(config).items():
    print(f"{key}: {value}")

In [ ]:
# ## 4. Neural Network Architecture
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(ActorCritic, self).__init__()
        
        # Actor network (policy)
        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, action_dim)
        )
        
        # Log std for action distribution (learnable parameter)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        
        # Critic network (value function)
        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
            nn.init.constant_(module.bias, 0.0)
    
    def forward(self, state):
        # Get action mean and value
        action_mean = self.actor(state)
        value = self.critic(state)
        return action_mean, value
    
    def get_action(self, state, deterministic=False):
        action_mean, value = self.forward(state)
        
        if deterministic:
            return action_mean, value, None, None
        
        # Create normal distribution
        action_std = torch.exp(self.log_std)
        dist = Normal(action_mean, action_std)
        
        # Sample action
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1, keepdim=True)
        
        return action, value, log_prob, dist.entropy().sum(dim=-1, keepdim=True)
    
    def evaluate_actions(self, states, actions):
        action_mean, value = self.forward(states)
        
        action_std = torch.exp(self.log_std)
        dist = Normal(action_mean, action_std)
        
        log_prob = dist.log_prob(actions).sum(dim=-1, keepdim=True)
        entropy = dist.entropy().sum(dim=-1, keepdim=True)
        
        return value, log_prob, entropy

In [ ]:
# ## 5. Rollout Buffer
class RolloutBuffer:
    def __init__(self, buffer_size, state_dim, action_dim, device):
        self.buffer_size = buffer_size
        self.device = device
        self.reset()
        
        self.states = torch.zeros((buffer_size, state_dim), dtype=torch.float32, device=device)
        self.actions = torch.zeros((buffer_size, action_dim), dtype=torch.float32, device=device)
        self.rewards = torch.zeros((buffer_size, 1), dtype=torch.float32, device=device)
        self.values = torch.zeros((buffer_size, 1), dtype=torch.float32, device=device)
        self.log_probs = torch.zeros((buffer_size, 1), dtype=torch.float32, device=device)
        self.dones = torch.zeros((buffer_size, 1), dtype=torch.float32, device=device)
    
    def reset(self):
        self.ptr = 0
        self.path_start_idx = 0
        self.advantages = None
        self.returns = None
    
    def add(self, state, action, reward, value, log_prob, done):
        self.states[self.ptr] = torch.FloatTensor(state).to(self.device)
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = torch.FloatTensor([reward]).to(self.device)
        self.values[self.ptr] = value
        self.log_probs[self.ptr] = log_prob
        self.dones[self.ptr] = torch.FloatTensor([done]).to(self.device)
        self.ptr += 1
    
    def compute_advantages(self, last_value, gamma, gae_lambda):
        """Compute GAE advantages"""
        self.advantages = torch.zeros_like(self.rewards, device=self.device)
        self.returns = torch.zeros_like(self.rewards, device=self.device)
        
        last_gae = 0
        for t in reversed(range(self.ptr)):
            if t == self.ptr - 1:
                next_value = last_value
                next_non_terminal = 1.0 - self.dones[t]
            else:
                next_value = self.values[t + 1]
                next_non_terminal = 1.0 - self.dones[t]
            
            delta = self.rewards[t] + gamma * next_value * next_non_terminal - self.values[t]
            last_gae = delta + gamma * gae_lambda * next_non_terminal * last_gae
            self.advantages[t] = last_gae
            self.returns[t] = self.advantages[t] + self.values[t]
    
    def get(self):
        # Normalize advantages
        advantages = self.advantages[:self.ptr]
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        return (
            self.states[:self.ptr],
            self.actions[:self.ptr],
            self.log_probs[:self.ptr],
            self.returns[:self.ptr],
            advantages
        )

In [ ]:
# ## 6. PPO Agent
class PPOAgent:
    def __init__(self, state_dim, action_dim, config):
        self.config = config
        self.state_dim = state_dim
        self.action_dim = action_dim
        
        # Networks
        self.policy = ActorCritic(state_dim, action_dim, config.hidden_dim).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=config.lr, eps=1e-5)
        
        # Buffer
        self.buffer = RolloutBuffer(config.rollout_buffer_size, state_dim, action_dim, device)
        
        # Training stats
        self.total_timesteps = 0
        self.clip_epsilon = config.clip_epsilon
        self.entropy_coef = config.entropy_coef
        
    def select_action(self, state, deterministic=False):
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action, value, log_prob, entropy = self.policy.get_action(state, deterministic)
        
        return (
            action.cpu().numpy()[0],
            value.cpu().item(),
            log_prob.cpu().item() if log_prob is not None else None,
            entropy.cpu().item() if entropy is not None else None
        )
    
    def update(self):
        # Compute advantages
        last_state = self.buffer.states[self.buffer.ptr - 1].unsqueeze(0)
        with torch.no_grad():
            _, last_value = self.policy(last_state)
        
        self.buffer.compute_advantages(last_value, self.config.gamma, self.config.gae_lambda)
        
        # Get data
        states, actions, old_log_probs, returns, advantages = self.buffer.get()
        
        # PPO update
        total_actor_loss = 0
        total_critic_loss = 0
        total_entropy = 0
        total_clip_fraction = 0
        n_updates = 0
        
        batch_size = self.buffer.ptr
        indices = np.arange(batch_size)
        
        for _ in range(self.config.ppo_epochs):
            np.random.shuffle(indices)
            
            for start in range(0, batch_size, self.config.mini_batch_size):
                end = start + self.config.mini_batch_size
                mb_indices = indices[start:end]
                
                mb_states = states[mb_indices]
                mb_actions = actions[mb_indices]
                mb_old_log_probs = old_log_probs[mb_indices]
                mb_returns = returns[mb_indices]
                mb_advantages = advantages[mb_indices]
                
                # Evaluate actions
                values, log_probs, entropy = self.policy.evaluate_actions(mb_states, mb_actions)
                
                # Actor loss (PPO clip)
                ratio = torch.exp(log_probs - mb_old_log_probs)
                surr1 = ratio * mb_advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * mb_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                
                # Critic loss
                critic_loss = F.mse_loss(values, mb_returns)
                
                # Entropy loss
                entropy_loss = -entropy.mean()
                
                # Total loss
                loss = actor_loss + self.config.value_coef * critic_loss + self.entropy_coef * entropy_loss
                
                # Optimize
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.optimizer.step()
                
                # Stats
                total_actor_loss += actor_loss.item()
                total_critic_loss += critic_loss.item()
                total_entropy += entropy.mean().item()
                
                # Clip fraction
                clip_fraction = ((ratio - 1).abs() > self.clip_epsilon).float().mean().item()
                total_clip_fraction += clip_fraction
                
                n_updates += 1
        
        # Decay parameters
        self.clip_epsilon = max(self.config.min_clip_epsilon, 
                               self.clip_epsilon * self.config.clip_epsilon_decay)
        self.entropy_coef = max(self.config.min_entropy_coef,
                               self.entropy_coef * self.config.entropy_decay)
        
        # Reset buffer
        self.buffer.reset()
        
        return {
            'actor_loss': total_actor_loss / n_updates,
            'critic_loss': total_critic_loss / n_updates,
            'entropy': total_entropy / n_updates,
            'clip_fraction': total_clip_fraction / n_updates
        }
    
    def save(self, path, metadata=None):
        checkpoint = {
            'policy_state_dict': self.policy.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'clip_epsilon': self.clip_epsilon,
            'entropy_coef': self.entropy_coef,
            'total_timesteps': self.total_timesteps,
            'metadata': metadata
        }
        torch.save(checkpoint, path)
    
    def load(self, path):
        checkpoint = torch.load(path, map_location=device, weights_only=False)  # Add this parameter
        self.policy.load_state_dict(checkpoint['policy_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.clip_epsilon = checkpoint.get('clip_epsilon', self.config.clip_epsilon)
        self.entropy_coef = checkpoint.get('entropy_coef', self.config.entropy_coef)
        self.total_timesteps = checkpoint.get('total_timesteps', 0)
        return checkpoint.get('metadata', {})

In [ ]:
# ## 7. Training Statistics Tracker
class StatsTracker:
    def __init__(self):
        self.episode_rewards = []
        self.episode_lengths = []
        self.actor_losses = []
        self.critic_losses = []
        self.entropies = []
        self.clip_fractions = []
        self.value_estimates = []
        self.moving_avg_rewards = []
        self.timesteps = []
        
    def add_episode(self, reward, length, timestep):
        self.episode_rewards.append(reward)
        self.episode_lengths.append(length)
        self.timesteps.append(timestep)
        
        # Compute moving average
        window = min(config.moving_avg_window, len(self.episode_rewards))
        moving_avg = np.mean(self.episode_rewards[-window:])
        self.moving_avg_rewards.append(moving_avg)
    
    def add_update_stats(self, stats):
        self.actor_losses.append(stats['actor_loss'])
        self.critic_losses.append(stats['critic_loss'])
        self.entropies.append(stats['entropy'])
        self.clip_fractions.append(stats['clip_fraction'])
    
    def get_latest_moving_avg(self):
        return self.moving_avg_rewards[-1] if self.moving_avg_rewards else 0
    
    def save_to_csv(self, path):
        import csv
        with open(path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['Episode', 'Reward', 'Length', 'MovingAvgReward', 'Timestep'])
            for i, (r, l, ma, t) in enumerate(zip(self.episode_rewards, 
                                                    self.episode_lengths, 
                                                    self.moving_avg_rewards,
                                                    self.timesteps)):
                writer.writerow([i+1, r, l, ma, t])

In [ ]:
# ## 8. Visualization Functions
def plot_training_progress(stats, save_path=None):
    """Create comprehensive training dashboard"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('PPO Training Progress - Walker2d-v4', fontsize=16, fontweight='bold')
    
    episodes = range(1, len(stats.episode_rewards) + 1)
    
    # 1. Episode Rewards
    ax = axes[0, 0]
    ax.plot(episodes, stats.episode_rewards, alpha=0.3, label='Episode Reward', color='blue')
    ax.plot(episodes, stats.moving_avg_rewards, label=f'{config.moving_avg_window}-Episode MA', 
            color='red', linewidth=2)
    ax.axhline(y=config.target_reward, color='green', linestyle='--', label='Target')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.set_title('Episode Rewards')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 2. Episode Lengths
    ax = axes[0, 1]
    ax.plot(episodes, stats.episode_lengths, alpha=0.6, color='purple')
    window = min(50, len(stats.episode_lengths))
    if window > 0:
        moving_avg_lengths = [np.mean(stats.episode_lengths[max(0, i-window):i+1]) 
                              for i in range(len(stats.episode_lengths))]
        ax.plot(episodes, moving_avg_lengths, color='orange', linewidth=2, label='50-Ep MA')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Length')
    ax.set_title('Episode Lengths')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # 3. Actor Loss
    if stats.actor_losses:
        ax = axes[0, 2]
        ax.plot(stats.actor_losses, color='darkred', alpha=0.7)
        ax.set_xlabel('Update Step')
        ax.set_ylabel('Loss')
        ax.set_title('Actor Loss (Policy Loss)')
        ax.grid(True, alpha=0.3)
    
    # 4. Critic Loss
    if stats.critic_losses:
        ax = axes[1, 0]
        ax.plot(stats.critic_losses, color='darkblue', alpha=0.7)
        ax.set_xlabel('Update Step')
        ax.set_ylabel('Loss')
        ax.set_title('Critic Loss (Value Loss)')
        ax.grid(True, alpha=0.3)
    
    # 5. Entropy
    if stats.entropies:
        ax = axes[1, 1]
        ax.plot(stats.entropies, color='green', alpha=0.7)
        ax.set_xlabel('Update Step')
        ax.set_ylabel('Entropy')
        ax.set_title('Policy Entropy')
        ax.grid(True, alpha=0.3)
    
    # 6. Clip Fraction
    if stats.clip_fractions:
        ax = axes[1, 2]
        ax.plot(stats.clip_fractions, color='orange', alpha=0.7)
        ax.axhline(y=0.1, color='red', linestyle='--', alpha=0.5, label='10% threshold')
        ax.set_xlabel('Update Step')
        ax.set_ylabel('Clip Fraction')
        ax.set_title('PPO Clip Fraction')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    # plt.show()

In [ ]:
def plot_final_summary(stats, save_path=None):
    """Create final summary plot"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle('Training Summary - Walker2d-v4', fontsize=16, fontweight='bold')
    
    episodes = range(1, len(stats.episode_rewards) + 1)
    
    # Rewards with multiple statistics
    ax = axes[0]
    ax.plot(episodes, stats.episode_rewards, alpha=0.2, color='blue', label='Raw Reward')
    ax.plot(episodes, stats.moving_avg_rewards, color='red', linewidth=2.5, 
            label=f'{config.moving_avg_window}-Episode MA')
    
    # Fill between for variance
    if len(stats.episode_rewards) >= config.moving_avg_window:
        rolling_std = [np.std(stats.episode_rewards[max(0, i-config.moving_avg_window):i+1]) 
                      for i in range(len(stats.episode_rewards))]
        upper = np.array(stats.moving_avg_rewards) + np.array(rolling_std)
        lower = np.array(stats.moving_avg_rewards) - np.array(rolling_std)
        ax.fill_between(episodes, lower, upper, alpha=0.2, color='red')
    
    ax.axhline(y=config.target_reward, color='green', linestyle='--', linewidth=2, label='Target')
    ax.set_xlabel('Episode', fontsize=12)
    ax.set_ylabel('Reward', fontsize=12)
    ax.set_title('Training Rewards Over Time')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Distribution of rewards
    ax = axes[1]
    ax.hist(stats.episode_rewards, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    ax.axvline(np.mean(stats.episode_rewards), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {np.mean(stats.episode_rewards):.2f}')
    ax.axvline(np.median(stats.episode_rewards), color='green', linestyle='--', 
               linewidth=2, label=f'Median: {np.median(stats.episode_rewards):.2f}')
    ax.set_xlabel('Reward', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('Reward Distribution')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Summary plot saved to {save_path}")
    
    plt.show()

In [ ]:
# ## 9. Video Recording Function
def record_video(env, agent, episode, save_path, max_steps=1000):
    """Record video of agent performance"""
    frames = []
    state, _ = env.reset()
    done = False
    total_reward = 0
    steps = 0
    
    while not done and steps < max_steps:
        # Render frame
        frame = env.render()
        frames.append(frame)
        
        # Get action
        action, _, _, _ = agent.select_action(state, deterministic=True)
        
        # Step
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1
    
    # Save video
    video_path = save_path / f"episode_{episode}_reward_{total_reward:.2f}.mp4"
    imageio.mimsave(str(video_path), frames, fps=30)
    print(f"Video saved: {video_path} (Reward: {total_reward:.2f})")
    
    return total_reward

In [ ]:
# ## 10. Main Training Loop
def train():
    print("\n" + "="*70)
    print("Starting PPO Training on Walker2d-v4")
    print("="*70 + "\n")
    
    # Create environments
    env = gym.make(config.env_name)
    video_env = gym.make(config.env_name, render_mode='rgb_array')
    
    # Get dimensions
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    
    print(f"State dimension: {state_dim}")
    print(f"Action dimension: {action_dim}")
    print(f"Action range: [{env.action_space.low[0]:.2f}, {env.action_space.high[0]:.2f}]\n")
    
    # Initialize agent and tracker
    agent = PPOAgent(state_dim, action_dim, config)
    stats = StatsTracker()
    
    # Best model tracking
    best_avg_reward = -np.inf
    episodes_without_improvement = 0
    
    # Training loop
    timestep = 0
    
    for episode in range(1, config.max_episodes + 1):
        state, _ = env.reset()
        episode_reward = 0
        episode_length = 0
        
        for step in range(config.max_timesteps):
            # Select action
            action, value, log_prob, entropy = agent.select_action(state)
            
            # Environment step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store in buffer
            agent.buffer.add(state, 
                           torch.FloatTensor(action).to(device),
                           reward, 
                           torch.FloatTensor([value]).to(device),
                           torch.FloatTensor([log_prob]).to(device),
                           done)
            
            episode_reward += reward
            episode_length += 1
            timestep += 1
            agent.total_timesteps = timestep
            
            state = next_state
            
            # Update if buffer is full
            if agent.buffer.ptr >= config.rollout_buffer_size:
                update_stats = agent.update()
                stats.add_update_stats(update_stats)
            
            if done:
                break
        
        # Record episode stats
        stats.add_episode(episode_reward, episode_length, timestep)
        current_avg = stats.get_latest_moving_avg()
        
        # Logging
        if episode % config.log_every == 0:
            print(f"Episode {episode}/{config.max_episodes} | "
                  f"Reward: {episode_reward:.2f} | "
                  f"Avg: {current_avg:.2f} | "
                  f"Length: {episode_length} | "
                  f"Timesteps: {timestep} | "
                  f"Clip ε: {agent.clip_epsilon:.3f} | "
                  f"Entropy coef: {agent.entropy_coef:.4f}")
        
        # Save checkpoint
        if episode % config.save_checkpoint_every == 0:
            checkpoint_path = CHECKPOINT_DIR / f"checkpoint_ep_{episode}.pt"
            metadata = {
                'episode': episode,
                'avg_reward': current_avg,
                'total_reward': episode_reward,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
            agent.save(checkpoint_path, metadata)
            print(f"Checkpoint saved: {checkpoint_path}")
        
        # Record video
        if episode % config.save_video_every == 0 or episode == 1:
            print(f"\nRecording video at episode {episode}...")
            record_video(video_env, agent, episode, VIDEO_DIR)
        
        # Update best model
        if current_avg > best_avg_reward:
            best_avg_reward = current_avg
            episodes_without_improvement = 0
            
            # Save best model
            best_model_path = BEST_MODEL_DIR / "best_model.pt"
            best_metadata = {
                'episode': episode,
                'best_avg_reward': best_avg_reward,
                'episode_reward': episode_reward,
                'timesteps': timestep,
                'config': vars(config),
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
            agent.save(best_model_path, best_metadata)
            
            # Save metadata separately
            with open(BEST_MODEL_DIR / "metadata.json", 'w') as f:
                json.dump(best_metadata, f, indent=4)
            
            # print(f"\n*** New best model! Avg reward: {best_avg_reward:.2f} ***\n")
        else:
            episodes_without_improvement += 1
        
        # Plot progress
        if episode % (config.save_checkpoint_every // 2) == 0:
            plot_path = PLOT_DIR / f"training_progress_ep_{episode}.png"
            plot_training_progress(stats, plot_path)
        
        # # Early stopping check
        # if current_avg >= config.target_reward:
        #     print(f"\n{'='*70}")
        #     print(f"Environment SOLVED! Target reward reached at episode {episode}")
        #     print(f"Average Reward: {current_avg:.2f}")
        #     print(f"{'='*70}\n")
        #     break
        
        # if episodes_without_improvement >= config.patience:
        #     print(f"\n{'='*70}")
        #     print(f"Early stopping: No improvement for {config.patience} episodes")
        #     print(f"Best average reward: {best_avg_reward:.2f}")
        #     print(f"{'='*70}\n")
        #     break
    
    # Final statistics
    print("\n" + "="*70)
    print("TRAINING COMPLETED")
    print("="*70)
    print(f"Total Episodes: {episode}")
    print(f"Total Timesteps: {timestep}")
    print(f"Best Average Reward: {best_avg_reward:.2f}")
    print(f"Final Average Reward: {current_avg:.2f}")
    print(f"Mean Reward: {np.mean(stats.episode_rewards):.2f}")
    print(f"Std Reward: {np.std(stats.episode_rewards):.2f}")
    print(f"Max Reward: {np.max(stats.episode_rewards):.2f}")
    print(f"Min Reward: {np.min(stats.episode_rewards):.2f}")
    print("="*70 + "\n")
    
    # Save final plots
    print("Generating final plots...")
    plot_training_progress(stats, PLOT_DIR / "final_training_progress.png")
    plot_final_summary(stats, PLOT_DIR / "final_summary.png")
    
    # Save training log
    stats.save_to_csv(BASE_PATH / "training_logs.csv")
    print(f"Training logs saved to: {BASE_PATH / 'training_logs.csv'}")
    
    # Record final video with best model
    print("\nRecording final video with best model...")
    best_model_path = BEST_MODEL_DIR / "best_model.pt"
    if best_model_path.exists():
        agent.load(best_model_path)
        record_video(video_env, agent, "FINAL", VIDEO_DIR, max_steps=1000)
    
    # Save final model
    final_model_path = BASE_PATH / "final_model.pt"
    final_metadata = {
        'total_episodes': episode,
        'total_timesteps': timestep,
        'final_avg_reward': current_avg,
        'best_avg_reward': best_avg_reward,
        'config': vars(config),
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    agent.save(final_model_path, final_metadata)
    print(f"Final model saved to: {final_model_path}")
    
    # Close environments
    env.close()
    video_env.close()
    
    return agent, stats

In [ ]:
# ## 11. Run Training
if __name__ == "__main__":
    # Train the agent
    trained_agent, training_stats = train()
    
    print("\n" + "="*70)
    print("All files saved to /kaggle/working/")
    print("="*70)
    print(f"- Checkpoints: {CHECKPOINT_DIR}")
    print(f"- Videos: {VIDEO_DIR}")
    print(f"- Plots: {PLOT_DIR}")
    print(f"- Best Model: {BEST_MODEL_DIR}")
    print(f"- Training Logs: {BASE_PATH / 'training_logs.csv'}")
    print("="*70 + "\n")

In [ ]:
import torch
import gymnasium as gym
import numpy as np
from datetime import datetime

def wrap_up_training():
    print("="*70)
    print("STARTING FINAL WRAP-UP (Video & Final Model Save)")
    print("="*70 + "\n")
    
    # 1. Setup Environments
    # We create the video_env with rgb_array for recording
    video_env = gym.make(config.env_name, render_mode='rgb_array')
    state_dim = video_env.observation_space.shape[0]
    action_dim = video_env.action_space.shape[0]
    
    # 2. Initialize Agent
    agent = PPOAgent(state_dim, action_dim, config)
    
    # 3. Load the best model
    best_model_path = BEST_MODEL_DIR / "best_model.pt"
    
    if best_model_path.exists():
        print(f"Loading weights from: {best_model_path}")
        # Using your internal load method which now has weights_only=False
        metadata = agent.load(best_model_path)
        
        # Extract metadata for the final save (falling back to config values if missing)
        best_avg = metadata.get('best_avg_reward', 0)
        ep = metadata.get('episode', config.max_episodes)
        ts = metadata.get('timesteps', agent.total_timesteps)
        
        print(f"Model loaded successfully. Best Avg Reward: {best_avg:.2f}")
    else:
        print(f"ERROR: No model found at {best_model_path}. Check your paths.")
        return

    # 4. Record final video with best model
    print(f"\nRecording final video (evaluating for 1000 steps)...")
    # record_video typically uses the agent to select actions; 
    # since we loaded the best weights into agent.policy, this will work.
    record_video(video_env, agent, "FINAL_WRAP_UP", VIDEO_DIR, max_steps=1000)
    print(f"Video saved to: {VIDEO_DIR}")
    
    # 5. Save final model
    # This fulfills your requirement to have a separate 'final_model.pt' file
    final_model_path = BASE_PATH / "final_model.pt"
    final_metadata = {
        'total_episodes': ep,
        'total_timesteps': ts,
        'final_avg_reward': best_avg,
        'best_avg_reward': best_avg,
        'config': vars(config),
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    agent.save(final_model_path, final_metadata)
    print(f"Final model saved to: {final_model_path}")
    
    # 6. Cleanup
    video_env.close()
    print("\n" + "="*70)
    print("WRAP-UP COMPLETE: You can now download your files.")
    print("="*70)

# Execute the function
if __name__ == "__main__":
    wrap_up_training()

In [ ]:
# Copy checkpoint to working directory
import shutil
from pathlib import Path

# Create directories
BEST_MODEL_DIR = Path("/kaggle/working/best_model")
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Copy uploaded checkpoint
shutil.copy('/kaggle/input/walker-v1/best_model.pt', 
            BEST_MODEL_DIR / 'best_model.pt')
shutil.copy('/kaggle/input/walker-v1/metadata.json',
            BEST_MODEL_DIR / 'metadata.json')

print("Checkpoint loaded successfully!")

In [ ]:
# ## Resume Training from Best Model

def resume_training(start_episode=3000, additional_episodes=2000):
    """Resume training from best model checkpoint"""
    print("\n" + "="*70)
    print("Resuming Training from Best Model")
    print("="*70 + "\n")
    
    # Create environments
    env = gym.make(config.env_name)
    video_env = gym.make(config.env_name, render_mode='rgb_array')
    
    # Get dimensions
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    
    # Initialize agent
    agent = PPOAgent(state_dim, action_dim, config)
    
    # Load best model
    best_model_path = BEST_MODEL_DIR / "best_model.pt"
    metadata = agent.load(best_model_path)
    print(f"Loaded best model from episode {metadata.get('episode', 'unknown')}")
    print(f"Previous best avg reward: {metadata.get('best_avg_reward', 'unknown'):.2f}\n")
    
    # Create new stats tracker
    stats = StatsTracker()
    
    # Best model tracking
    best_avg_reward = metadata.get('best_avg_reward', -np.inf)
    
    # Training loop
    timestep = agent.total_timesteps
    total_episodes = start_episode + additional_episodes
    
    for episode in range(start_episode + 1, total_episodes + 1):
        state, _ = env.reset()
        episode_reward = 0
        episode_length = 0
        
        for step in range(config.max_timesteps):
            # Select action
            action, value, log_prob, entropy = agent.select_action(state)
            
            # Environment step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store in buffer
            agent.buffer.add(state, 
                           torch.FloatTensor(action).to(device),
                           reward, 
                           torch.FloatTensor([value]).to(device),
                           torch.FloatTensor([log_prob]).to(device),
                           done)
            
            episode_reward += reward
            episode_length += 1
            timestep += 1
            agent.total_timesteps = timestep
            
            state = next_state
            
            # Update if buffer is full
            if agent.buffer.ptr >= config.rollout_buffer_size:
                update_stats = agent.update()
                stats.add_update_stats(update_stats)
            
            if done:
                break
        
        # Record episode stats
        stats.add_episode(episode_reward, episode_length, timestep)
        current_avg = stats.get_latest_moving_avg()
        
        # Logging
        if episode % config.log_every == 0:
            print(f"Episode {episode}/{total_episodes} | "
                  f"Reward: {episode_reward:.2f} | "
                  f"Avg: {current_avg:.2f} | "
                  f"Length: {episode_length} | "
                  f"Timesteps: {timestep} | "
                  f"Clip ε: {agent.clip_epsilon:.3f} | "
                  f"Entropy coef: {agent.entropy_coef:.4f}")
        
        # Save checkpoint
        if episode % config.save_checkpoint_every == 0:
            checkpoint_path = CHECKPOINT_DIR / f"checkpoint_ep_{episode}_resumed.pt"
            metadata_ckpt = {
                'episode': episode,
                'avg_reward': current_avg,
                'total_reward': episode_reward,
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'resumed_training': True
            }
            agent.save(checkpoint_path, metadata_ckpt)
            print(f"Checkpoint saved: {checkpoint_path}")
        
        # Record video
        if episode % config.save_video_every == 0:
            print(f"\nRecording video at episode {episode}...")
            record_video(video_env, agent, episode, VIDEO_DIR)
        
        # Update best model
        if current_avg > best_avg_reward:
            best_avg_reward = current_avg
            
            # Save best model
            best_model_path = BEST_MODEL_DIR / "best_model_resumed.pt"
            best_metadata = {
                'episode': episode,
                'best_avg_reward': best_avg_reward,
                'episode_reward': episode_reward,
                'timesteps': timestep,
                'config': vars(config),
                'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'resumed_training': True,
                'original_best': metadata.get('best_avg_reward', 'unknown')
            }
            agent.save(best_model_path, best_metadata)
            
            # Save metadata
            with open(BEST_MODEL_DIR / "metadata_resumed.json", 'w') as f:
                json.dump(best_metadata, f, indent=4)
            
            # print(f"\n*** New best model! Avg reward: {best_avg_reward:.2f} ***\n")
        
        # Plot progress
        if episode % (config.save_checkpoint_every // 2) == 0:
            plot_path = PLOT_DIR / f"resumed_training_progress_ep_{episode}.png"
            plot_training_progress(stats, plot_path)
    
    # Final statistics
    print("\n" + "="*70)
    print("RESUMED TRAINING COMPLETED")
    print("="*70)
    print(f"Total Episodes: {episode}")
    print(f"Additional Episodes: {additional_episodes}")
    print(f"Total Timesteps: {timestep}")
    print(f"Best Average Reward: {best_avg_reward:.2f}")
    print(f"Final Average Reward: {current_avg:.2f}")
    print("="*70 + "\n")
    
    # Save final plots
    plot_training_progress(stats, PLOT_DIR / "resumed_final_progress.png")
    plot_final_summary(stats, PLOT_DIR / "resumed_final_summary.png")
    
    # Save training log
    stats.save_to_csv(BASE_PATH / "resumed_training_logs.csv")
    
    # Close environments
    env.close()
    video_env.close()
    
    return agent, stats

# Run resumed training for 3000 more episodes
resumed_agent, resumed_stats = resume_training(start_episode=3000, additional_episodes=3000)

In [ ]:
# ## 12. Evaluation Function (Optional)
def evaluate_model(agent, num_episodes=10):
    """Evaluate trained model"""
    print(f"\nEvaluating model over {num_episodes} episodes...")
    
    eval_env = gym.make(config.env_name)
    eval_rewards = []
    
    for ep in range(num_episodes):
        state, _ = eval_env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            action, _, _, _ = agent.select_action(state, deterministic=True)
            state, reward, terminated, truncated, _ = eval_env.step(action)
            episode_reward += reward
            done = terminated or truncated
        
        eval_rewards.append(episode_reward)
        print(f"Eval Episode {ep+1}: Reward = {episode_reward:.2f}")
    
    eval_env.close()
    
    print(f"\nEvaluation Results:")
    print(f"Mean Reward: {np.mean(eval_rewards):.2f} ± {np.std(eval_rewards):.2f}")
    print(f"Max Reward: {np.max(eval_rewards):.2f}")
    print(f"Min Reward: {np.min(eval_rewards):.2f}")
    
    return eval_rewards

# To evaluate the best model after training:
evaluate_model(trained_agent, num_episodes=10)

In [ ]:
def evaluate_model_resumed(num_episodes=20):
    """Evaluate the resumed best model"""
    print(f"\nEvaluating resumed best model over {num_episodes} episodes...")
    print("="*70)
    
    # Create environment
    eval_env = gym.make(config.env_name)
    
    # Get dimensions
    state_dim = eval_env.observation_space.shape[0]
    action_dim = eval_env.action_space.shape[0]
    
    # Initialize agent
    agent = PPOAgent(state_dim, action_dim, config)
    
    # Load resumed best model
    best_model_path = BEST_MODEL_DIR / "best_model_resumed.pt"
    
    if not best_model_path.exists():
        print(f"Error: {best_model_path} not found!")
        print("Available models:")
        for f in BEST_MODEL_DIR.glob("*.pt"):
            print(f"  - {f.name}")
        eval_env.close()
        return None
    
    metadata = agent.load(best_model_path)
    print(f"Loaded model from episode: {metadata.get('episode', 'unknown')}")
    print(f"Training avg reward: {metadata.get('best_avg_reward', 'unknown'):.2f}")
    print(f"Original best: {metadata.get('original_best', 'unknown')}\n")
    
    eval_rewards = []
    eval_lengths = []
    
    for ep in range(num_episodes):
        state, _ = eval_env.reset()
        episode_reward = 0
        episode_length = 0
        done = False
        
        while not done:
            action, _, _, _ = agent.select_action(state, deterministic=True)
            state, reward, terminated, truncated, _ = eval_env.step(action)
            episode_reward += reward
            episode_length += 1
            done = terminated or truncated
        
        eval_rewards.append(episode_reward)
        eval_lengths.append(episode_length)
        print(f"Eval Episode {ep+1:2d}: Reward = {episode_reward:8.2f}, Length = {episode_length:4d}")
    
    eval_env.close()
    
    print("\n" + "="*70)
    print("EVALUATION RESULTS")
    print("="*70)
    print(f"Number of Episodes: {num_episodes}")
    print(f"Mean Reward:        {np.mean(eval_rewards):8.2f} ± {np.std(eval_rewards):.2f}")
    print(f"Median Reward:      {np.median(eval_rewards):8.2f}")
    print(f"Max Reward:         {np.max(eval_rewards):8.2f}")
    print(f"Min Reward:         {np.min(eval_rewards):8.2f}")
    print(f"Mean Length:        {np.mean(eval_lengths):8.2f}")
    print("="*70 + "\n")
    
    # Plot evaluation results
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Rewards
    ax = axes[0]
    ax.plot(range(1, num_episodes+1), eval_rewards, marker='o', linewidth=2, markersize=6)
    ax.axhline(np.mean(eval_rewards), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(eval_rewards):.2f}')
    ax.set_xlabel('Episode', fontsize=12)
    ax.set_ylabel('Reward', fontsize=12)
    ax.set_title('Evaluation Episode Rewards', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Distribution
    ax = axes[1]
    ax.hist(eval_rewards, bins=15, alpha=0.7, color='skyblue', edgecolor='black')
    ax.axvline(np.mean(eval_rewards), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(eval_rewards):.2f}')
    ax.axvline(np.median(eval_rewards), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(eval_rewards):.2f}')
    ax.set_xlabel('Reward', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('Reward Distribution', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    eval_plot_path = PLOT_DIR / "resumed_model_evaluation.png"
    plt.savefig(eval_plot_path, dpi=150, bbox_inches='tight')
    print(f"Evaluation plot saved to: {eval_plot_path}")
    plt.show()
    
    return eval_rewards, eval_lengths

# Run evaluation
eval_rewards, eval_lengths = evaluate_model_resumed(num_episodes=20)

In [ ]:
import shutil
import os

# Zip the entire working directory (or a specific folder)
# Output: /kaggle/working/training_results.zip
shutil.make_archive('training_results_pt2', 'zip', '/kaggle/working/')

print("Zip file created! Look for 'training_results_pt2.zip' in the sidebar.")